In [5]:
import pandas as pd
import numpy as np
import imblearn
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Input, Embedding, Bidirectional, LSTM, Dropout, Dense, GlobalMaxPooling1D, GlobalAveragePooling1D, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2
from imblearn.over_sampling import SMOTE
from sklearn.utils.class_weight import compute_class_weight


In [6]:
df = pd.read_csv("nepal_data_vader_labeled2.csv")
X_text = df['lemmatized_text'].astype(str)
y_text = df['VADER_sentiment']


In [8]:
# Label encode
le = LabelEncoder()
y = le.fit_transform(y_text)

# Split
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text, y, test_size=0.2, random_state=42, stratify=y
)

In [9]:
# Tokenize
tokenizer = Tokenizer(num_words=20000, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_text)

X_train_seq = tokenizer.texts_to_sequences(X_train_text)
X_test_seq = tokenizer.texts_to_sequences(X_test_text)

X_train_pad = pad_sequences(X_train_seq, maxlen=200, padding='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=200, padding='post')


In [10]:
# Build model
input_layer = Input(shape=(200,))
embedding = Embedding(input_dim=20000, output_dim=256)(input_layer)  # increased embedding
bilstm = Bidirectional(LSTM(512, return_sequences=True))(embedding) # increased LSTM units
avg_pool = GlobalAveragePooling1D()(bilstm)
max_pool = GlobalMaxPooling1D()(bilstm)
conc = Concatenate()([avg_pool, max_pool])
dense_feature = Dense(256, activation='relu', kernel_regularizer=l2(0.01), name='feature_layer')(conc)
dropout = Dropout(0.3)(dense_feature) # reduced dropout
output = Dense(3, activation='softmax')(dropout)

bilstm_classifier = Model(inputs=input_layer, outputs=output)
bilstm_classifier.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(enumerate(class_weights))
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
bilstm_classifier.fit(
    X_train_pad, y_train, epochs=15, batch_size=64, validation_data=(X_test_pad, y_test), verbose=1, class_weight=class_weight_dict, callbacks=[early_stopping]
)


Epoch 1/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 407s 4s/step - accuracy: 0.4949 - loss: 1.6891 - val_accuracy: 0.6763 - val_loss: 0.8149
Epoch 2/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 493s 5s/step - accuracy: 0.7825 - loss: 0.5956 - val_accuracy: 0.8453 - val_loss: 0.5081
Epoch 3/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 537s 5s/step - accuracy: 0.9233 - loss: 0.2932 - val_accuracy: 0.8485 - val_loss: 0.4401
Epoch 4/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 546s 6s/step - accuracy: 0.9670 - loss: 0.1567 - val_accuracy: 0.8504 - val_loss: 0.4942
Epoch 5/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 531s 5s/step - accuracy: 0.9776 - loss: 0.1151 - val_accuracy: 0.8524 - val_loss: 0.4713
Epoch 6/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 511s 5s/step - accuracy: 0.9858 - loss: 0.0835 - val_accuracy: 0.8395 - val_loss: 0.6125


In [12]:
# Feature extractor
feature_extractor = Model(
    inputs=bilstm_classifier.input,
    outputs=bilstm_classifier.get_layer('feature_layer').output
)

X_train_features = feature_extractor.predict(X_train_pad)
X_test_features = feature_extractor.predict(X_test_pad)

scaler = StandardScaler()
X_train_features = scaler.fit_transform(X_train_features)
X_test_features = scaler.transform(X_test_features)

194/194 ━━━━━━━━━━━━━━━━━━━━ 184s 939ms/step
49/49 ━━━━━━━━━━━━━━━━━━━━ 39s 794ms/step


In [13]:
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC

params = {
    'C': [0.1, 1, 10],
    'kernel': ['linear', 'rbf'],
    'gamma': ['scale', 'auto']
}

grid = GridSearchCV(
    SVC(probability=True, class_weight='balanced'),
    params,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1
)

grid.fit(X_train_features, y_train)

y_test_pred = grid.predict(X_test_features)

print("Best Params:", grid.best_params_)
print("Accuracy:", accuracy_score(y_test, y_test_pred))
print(classification_report(y_test, y_test_pred, target_names=le.classes_))

Best Params: {'C': 0.1, 'gamma': 'scale', 'kernel': 'linear'}
Accuracy: 0.8549323017408124
              precision    recall  f1-score   support

    negative       0.83      0.89      0.86       544
     neutral       0.90      0.84      0.87       505
    positive       0.85      0.83      0.84       502

    accuracy                           0.85      1551
   macro avg       0.86      0.85      0.85      1551
weighted avg       0.86      0.85      0.85      1551



In [ ]:
# --- Added SHAP Explainability ---
import shap

print("\nStarting SHAP explanation on a sample of around 200 rows of text...")
# Filter out texts with only 1 word, which crash the SHAP Text masker's hierarchical clustering
import re
valid_texts = []
for text in X_test_text.astype(str):
    if not pd.isna(text) and text.strip() != "":
        # Ensure the text has at least 2 words (SHAP PartitionExplainer needs >= 2 words to cluster)
        if len(re.findall(r"\w+", text)) >= 2:
            valid_texts.append(text)

shap_sample_texts = valid_texts[:200]

def predict_proba_text(texts):
    # Determine the shape/type of texts being passed by SHAP
    if isinstance(texts, np.ndarray):
        texts = texts.tolist()
    
    seqs = tokenizer.texts_to_sequences(texts)
    pads = pad_sequences(seqs, maxlen=200, padding='post')
    # Use explicit batch_size to prevent out-of-memory errors
    features = feature_extractor.predict(pads, batch_size=32, verbose=0)
    features_scaled = scaler.transform(features)
    return grid.predict_proba(features_scaled)

# Create an explainer using a text masker
masker = shap.maskers.Text(r"\W+")
explainer = shap.Explainer(predict_proba_text, masker, output_names=list(le.classes_))

# Calculate SHAP values
# Adding max_evals to speed up execution and reduce memory overhead for 200 samples
try:
    shap_values = explainer(shap_sample_texts, max_evals=300)

    # Generate an HTML report and save it
    print("Generating SHAP explanation HTML...")
    shap_html = shap.plots.text(shap_values, display=False)
    with open("C:\\Users\\madhu\\OneDrive\\Documents\\New folder\\hybrid model\\shap_explanation.html", "w", encoding="utf-8") as f:
        f.write(shap_html)
    print("SHAP explanation successfully saved to 'shap_explanation.html'")
except Exception as e:
    print(f"SHAP encountered an error: {e}")
    import traceback
    traceback.print_exc()



Starting SHAP explanation on a sample of around 200 rows of text...


  0%|          | 0/298 [00:00<?, ?it/s]

PartitionExplainer explainer:   0%|▎                                                           | 1/200 [00:00<?, ?it/s]

  0%|          | 0/272 [00:00<?, ?it/s]

PartitionExplainer explainer:   4%|██                                                  | 8/200 [01:53<14:28,  4.52s/it]

...

PartitionExplainer explainer:  79%|███████████████████████████████████████▌          | 158/200 [31:34<05:10,  7.39s/it]

  0%|          | 0/72 [00:00<?, ?it/s]

PartitionExplainer explainer:  80%|████████████████████████████████████████          | 160/200 [31:46<04:15,  6.39s/it]

  0%|          | 0/90 [00:00<?, ?it/s]

PartitionExplainer explainer:  80%|████████████████████████████████████████▎         | 161/200 [31:56<04:54,  7.56s/it]